# Analyzing Images

In this notebook, we send an image to an OpenAI model and ask it to identify the visible ingredients in a fridge.

We also ask for rough amounts, so the response includes estimates like milliliters, grams, or pieces.

Docs:
- [Images and vision](https://developers.openai.com/api/docs/guides/images-vision)
- [File inputs](https://developers.openai.com/api/docs/guides/file-inputs)

## Setup

In [12]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Image, display
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(override=True)
client = OpenAI()

## Load the image

The fridge image is stored in the local `files` folder. 

Supported image file types:
- PNG (`.png`)
- JPEG (`.jpeg`, `.jpg`)
- WEBP (`.webp`)
- Non-animated GIF (`.gif`)

In [ ]:
FRIDGE_IMAGE_PATH = Path("files/fridge.png")
display(Image(filename=str(FRIDGE_IMAGE_PATH)))

## Defining the model

Besides text, modern models can also process image inputs and anlayze them - this is a capability known as *vision*. 

A model that can process or generate more than one kind of data, such as text, images, audio, or video, is called a *multimodal model*.

[All OpenAI Models](https://developers.openai.com/api/docs/models)

First Vision Model: [GPT-4o](https://developers.openai.com/api/docs/models/gpt-4o) (Released May 2024)

Current Frontier Model: [GPT-5.5](https://developers.openai.com/api/docs/models/gpt-5.5)

In [13]:
MODEL = "gpt-5.4-mini-2026-03-17"

## Define the data we want back

A Pydantic model gives the response a clear shape. Each ingredient has a name, a rough amount, a unit, and optional notes for uncertainty.

In [14]:
class Ingredient(BaseModel):
    name: str
    amount: float | None
    unit: str | None
    notes: str | None


class FridgeAnalysis(BaseModel):
    ingredients: list[Ingredient]

## Define instructions

In [15]:
INSTRUCTIONS = (
    "Analyze the image and identify visible ingredients. "
    "Return one item per ingredient or package. "
    "Estimate rough amounts. "
    "Use grams for solid loose ingredients, milliliters for liquids, "
    "and pieces for countable items or packages. "
    "Use null for amount and unit if no reasonable estimate is possible. "
    "Use short notes for uncertainty, packaging, or partially hidden items."
)

## Option 1 - Web URL

If an image already lives online, send it with an `image_url`. The URL must be public so the API can fetch it.

- Analyze Image [docs](https://developers.openai.com/api/docs/guides/images-vision?format=url#analyze-images)
- Detail level [docs](https://developers.openai.com/api/docs/guides/images-vision#choose-an-image-detail-level)

In [16]:
FRIDGE_IMAGE_URL = (
    "https://raw.githubusercontent.com/LukasLechnerDev/"
    "AI-Engineering-Foundations-Labs/main/"
    "3-interacting-with-llm-apis/files/fridge.png"
)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": FRIDGE_IMAGE_URL,
                    "detail": "high",
                },
            ],
        }  # type: ignore
    ],
    text_format=FridgeAnalysis,
)

fridge_analysis = response.output_parsed

if fridge_analysis is not None:
    print(fridge_analysis.model_dump_json(indent=2))

{
  "ingredients": [
    {
      "name": "blueberries",
      "amount": 150.0,
      "unit": "grams",
      "notes": "small clamshell on top shelf"
    },
    {
      "name": "Good Culture cottage cheese",
      "amount": 1.0,
      "unit": "piece",
      "notes": "tub"
    },
    {
      "name": "Siggi's yogurt",
      "amount": 1.0,
      "unit": "piece",
      "notes": "tub"
    },
    {
      "name": "jarred sauce/spread",
      "amount": 1.0,
      "unit": "piece",
      "notes": "partially hidden jar, top shelf"
    },
    {
      "name": "jarred sauce/spread",
      "amount": 1.0,
      "unit": "piece",
      "notes": "yellow lid jar, top shelf"
    },
    {
      "name": "small container of shredded cheese",
      "amount": 50.0,
      "unit": "grams",
      "notes": "clear container, top shelf"
    },
    {
      "name": "avocado",
      "amount": 1.0,
      "unit": "piece",
      "notes": "top shelf"
    },
    {
      "name": "jarred condiment",
      "amount": 1.0,
      "u

## Option 2 - Send a base64 encoded image

For a local one-off image, base64 is the simplest option. We send the image bytes directly in the request.

More information in the [docs](https://developers.openai.com/api/docs/guides/images-vision?format=base64-encoded#analyze-images)

In [ ]:
# Function to encode the image
def encode_image(image_path):
    # image is opened in raw binary ("rb") mode and then encoded to base64
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


fridge_base64 = encode_image(FRIDGE_IMAGE_PATH)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{fridge_base64}",
                    "detail": "high",
                },
            ],
        }  # type: ignore
    ],
    text_format=FridgeAnalysis,
)

fridge_analysis = response.output_parsed

if fridge_analysis is not None:
    print(fridge_analysis.model_dump_json(indent=2))

## Option 3 - File upload

If you want to reuse the same image in multiple requests, upload it once and then reference the returned file id.

Files get uploaded to: https://platform.openai.com/storage

More information in the [docs](https://developers.openai.com/api/docs/guides/images-vision?format=file#analyze-images)

In [ ]:
print("Start uploading the image to the OpenAI API...")

with FRIDGE_IMAGE_PATH.open("rb") as image_file:
    uploaded_image = client.files.create(
        file=image_file,
        purpose="vision",
    )

print("Upload completed. File ID:", uploaded_image.id)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "file_id": uploaded_image.id,
                    "detail": "high",
                },
            ],
        }  # type: ignore
    ],
    text_format=FridgeAnalysis,
)

fridge_analysis = response.output_parsed

if fridge_analysis is not None:
    print(fridge_analysis.model_dump_json(indent=2))

## Takeaways

- Use a web URL when the image is already public.
- Use base64 when you have a local image and only need one request.
- Use file upload when you want to reuse the same image across multiple requests.

## Hungry? Let's generate some recipe ideas!

In [17]:
from IPython.display import Markdown

recipe_response = client.responses.create(
    model=MODEL,
    instructions=(
        "Suggest 3 practical recipe ideas using the fridge ingredients. "
        "Use the listed ingredients as the main ingredients. "
        "You may assume basic pantry staples like salt, pepper, oil, and water. "
        "Return the ideas as concise markdown."
    ),
    input=fridge_analysis.model_dump_json(indent=2),
)

display(Markdown(recipe_response.output_text))

### 1) Berry Yogurt Cottage Cheese Bowl
- **Use:** blueberries, strawberries, Siggi’s yogurt, Good Culture cottage cheese, citrus fruit
- **How:** Spoon yogurt and cottage cheese into a bowl, top with blueberries and strawberries, and add orange/citrus segments.  
- **Optional:** drizzle with a little honey or sprinkle with salt if you want the fruit flavor to pop.

### 2) Veggie Egg Scramble with Avocado Toast
- **Use:** eggs, chopped vegetable mix, bell peppers, leafy greens, avocado, bread roll or bun, shredded cheese, jarred sauce/spread
- **How:** Sauté the chopped vegetables and greens in oil, scramble in the eggs, and melt in shredded cheese. Serve on toasted bun/roll with sliced avocado and a little sauce/spread.

### 3) Bacon, Greens & Avocado Breakfast Sandwich
- **Use:** bacon, eggs, leafy greens or spinach, avocado, bread roll or bun, shredded cheese, jarred condiment
- **How:** Cook the bacon and an egg to your liking, then layer on the bun with greens, avocado, cheese, and a smear of condiment.  
- **Optional:** add sliced bell pepper for crunch.